# TFT - model selection

Hyperparameters were searched separately for each feature set with Optuna. Each step has its own study, `tft_d1`, `tft_d2` or `tft_d3`, stored in `results/tft/<step>/study.db`. The objective is validation MAE, minimized. A trial trains one Temporal Fusion Transformer on seed 42 for at most 40 epochs. Early stopping watches `val_MAE` with patience 8 and a minimum improvement of 0.1, and a median pruner can stop a trial after 10 epochs once 30 trials have reported. The sampler is multivariate TPE with 60 random startup trials. The horizon is 24 hours, the loss is the median quantile, the optimizer is AdamW and the batch size is 128. `hidden_continuous_size` is half of `hidden_size` and is not searched.

The searched ranges are:
* `max_encoder_length` in {24, 48, 72}, 
* `hidden_size` in {16, 32, 64, 128}, 
* `attention_head_size` in {1, 2, 4}, 
* `lstm_layers` in {1, 2, 3}, 
* `dropout` from 0.05 to 0.4, 
* `learning_rate` from 1e-4 to 5e-3 on a log scale, 
* `weight_decay` from 1e-6 to 1e-2 on a log scale,
* `gradient_clip_val` from 0.01 to 1.0 on a log scale.

The five complete trials with the lowest Optuna score are then trained again, on seeds 1–5, and scored by MAE against `PL_price` on the validation set, in EUR/MWh. For each feature set the three Optuna figures come first and the seed table follows. Lower is better. The summary places the Optuna score next to the mean and the standard deviation of these five seeds, and the last section names the trial carried into the backtest.


In [ ]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.config import BASE_DIR
from src.utils.evaluation import score_forecast
from src.utils.file_io import load_final_dataset

rows = []
for step in ["d1", "d2", "d3"]:
    prices = load_final_dataset("tft", step, "val")["PL_price"]
    top = json.loads((BASE_DIR / "results" / "tft" / step / "top5.json").read_text())
    tune = {trial["number"]: trial["value"] for trial in top["trials"]}
    val_dir = BASE_DIR / "results" / "tft" / step / "val"

    for path in sorted(val_dir.glob("trial_*.csv")):
        forecast = pd.read_csv(path, index_col=0, parse_dates=True)
        number = int(path.stem.removeprefix("trial_"))
        maes = {
            seed: score_forecast(prices, forecast[[seed]], metrics=["MAE"]).iloc[0]
            for seed in forecast.columns
        }
        rows.append(
            {
                "step": step,
                "trial": number,
                "tune": tune[number],
                **maes,
            }
        )

seeds = ["1", "2", "3", "4", "5"]
scores = pd.DataFrame(rows)
scores["mean"] = scores[seeds].mean(axis=1)
scores["std"] = scores[seeds].std(axis=1)


def seed_table(step):
    return (
        scores.loc[scores["step"] == step]
        .sort_values("mean")
        .set_index("trial")[seeds]
        .round(3)
    )

## Step - d1

### Trial history
<img src="../../img/optuna/d1/trial_history.png" width="800" height="300">

### Hyperparameter importance
<img src="../../img/optuna/d1/hp_importance.png" width="800" height="300">

### Hyperparameters of the top 5
<img src="../../img/optuna/d1/hp_top5.png" width="800" height="300">

The table is ordered by the five-seed mean, so trial 114 is the first row and trial 117 the last. In every row seed 3 is the lowest of the five seeds, and in the first four rows it lies between 18.594 and 18.841. The other seeds of trial 114 lie between 18.839 and 19.653. Trial 117 is the last row, and its seed 4 at 20.692 is the highest cell in the table.


In [2]:
seed_table("d1")

,1,2,3,4,5
trial,,,,,
114,19.197,19.653,18.595,18.839,18.865
119,19.035,19.910,18.594,19.100,18.962
84,19.880,19.593,18.603,19.373,19.109
85,20.021,20.537,18.841,19.345,18.953
117,19.706,20.476,18.912,20.692,18.915


## Step - d2

### Trial history
<img src="../../img/optuna/d2/trial_history.png" width="800" height="300">

### Hyperparameter importance
<img src="../../img/optuna/d2/hp_importance.png" width="800" height="300">

### Hyperparameters of the top 5
<img src="../../img/optuna/d2/hp_top5.png" width="800" height="300">

Trial 75 is the first row and trial 119 the last. Trials 75 and 64 keep every seed between 14.545 and 16.280. Trial 78 holds the lowest cell in the table, seed 3 at 13.885, next to seeds 1, 2 and 5 at 16.429, 16.348 and 16.139. Trial 119 is the last row, and its seed 1 at 18.389 is the highest cell in the table, while the other four seeds of that row stay between 14.233 and 15.829.


In [3]:
seed_table("d2")

,1,2,3,4,5
trial,,,,,
75,15.747,16.165,14.545,14.788,16.248
64,15.702,15.933,14.743,14.995,16.280
78,16.429,16.348,13.885,14.916,16.139
87,16.398,15.008,15.243,15.749,16.632
119,18.389,15.719,15.829,14.233,15.631


## Step - d3

### Trial history
<img src="../../img/optuna/d3/trial_history.png" width="800" height="300">

### Hyperparameter importance
<img src="../../img/optuna/d3/hp_importance.png" width="800" height="300">

### Hyperparameters of the top 5
<img src="../../img/optuna/d3/hp_top5.png" width="800" height="300">

Trial 99 is the first row. Seeds 1, 2, 4 and 5 of that row lie between 16.064 and 16.759, and seed 3 is 18.405. Trial 76 is the last row, and its seed 3 at 21.863 is the highest cell in the table. The other four seeds of trial 76 lie between 16.834 and 19.017, and seed 3 is also the highest seed of trial 69, at 19.317.


In [4]:
seed_table("d3")

,1,2,3,4,5
trial,,,,,
99,16.759,16.250,18.405,16.666,16.064
69,16.907,16.341,19.317,15.468,17.475
84,18.075,18.485,16.329,18.124,16.836
112,17.838,17.660,18.266,16.004,19.508
76,18.464,17.638,21.863,19.017,16.834


## Summary

The rows are sorted by `mean` within each step. `tune` is the best validation MAE Optuna recorded while training seed 42, and in every row it sits below `mean`, the average of seeds 1–5 scored at the end of a fresh training. On d1 the five means follow the same order as `tune`, from 19.030 for trial 114 (std 0.409) to 19.740 for trial 117 (std 0.839). On d2 the first three means are 15.498, 15.531 and 15.543, and trial 119, with `tune` 14.808, is last on the mean at 15.960 with std 1.506. On d3 the means open up, from 16.829 for trial 99 (std 0.927) to 18.763 for trial 76 (std 1.920).


In [ ]:
summary = scores.set_index(["step", "trial"])[["tune", "mean", "std"]].sort_values(
    ["step", "mean"]
)
summary.round(3)

tune    mean    std
step trial                       
d1   114    17.641  19.030  0.409
     119    17.690  19.120  0.483
     84     17.714  19.312  0.487
     85     17.717  19.539  0.724
     117    17.721  19.740  0.839
d2   75     14.680  15.498  0.788
     64     14.825  15.531  0.644
     78     14.900  15.543  1.110
     87     14.876  15.806  0.705
     119    14.808  15.960  1.506
d3   99     14.529  16.829  0.927
     69     14.599  17.102  1.444
     84     14.612  17.570  0.932
     112    14.748  17.855  1.261
     76     14.846  18.763  1.920

## Conclusions

The ranking follows `mean`, so the first row of each step is the configuration taken forward: trial 114 for d1, trial 75 for d2 and trial 99 for d3. On d1 that row also has the smallest std of the step, 0.409. On d3 trial 99 has std 0.927, level with trial 84 at 0.932, and its mean is the lowest by the margin visible in the table, 16.829 against 17.102 for the next row. On d2 trial 64 is the next row, mean 15.531 against 15.498, with a smaller std, 0.644 against 0.788, so the two stay close and the mean still places trial 75 first. These three trial numbers are the `--trial` argument of the backtest.
